# Chapter 7: Bind an answer to a typed reference

Chapter 7 separates answer correctness, abstention, output validity and source attribution. The live answer tests of this run were scored by `answer_score` in `agent_reporting.py`. This notebook checks that scorer on four constructed responses before its live verdicts are read in the next notebooks. Answer correctness, abstention, output validity and source attribution are separate checks. Verify that wrong values and malformed responses do not pass because they contain familiar tokens.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` ran numerical discovery with three new optimizer seeds (74101 to 74103), one live structural proposal and a fresh 200,000-observation confirmation sample. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The population and frozen logistic predictor are those of the chapter companions; the seeds, proposal and confirmation sample are new, so these numbers should not be combined with the historical run's.

## Setup and source checks

The setup cell locates the data root, the directory whose relative layout the evidence records use. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package. `R` is this run's folder under the data root, and `read` loads a JSON file from it.

`reference.json` was exported from the run's regional results and records their SHA-256 digest. The final assertion recomputes that digest, so every value used below comes from the unchanged results of this run. This notebook reads recorded outputs; rerunning it makes no language-model call.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Check the answer scorer

A response must be JSON with exactly two fields, a numeric or null `answer` and a list of source identifiers. `answer_score(record, task, access)` returns `unscorable` for any other form. When source records were supplied, the expected answer is the task's value. A numeric answer within $10^{-6}$ of it is `correct_answer`. When sources were withheld, or the task asks for an unmeasured result, the expected answer is null and a null answer is `correct_abstention`. A separate flag, `provenance_correct`, checks that the cited identifiers equal the required set. The four constructed cases cover a correct value, a wrong value that cites the right source, a null answer without access and a Boolean answer.

In [2]:
import sys
sys.path.insert(0,str(LIVE))
from agent_reporting import answer_score
task=dict(answer=.25,sources=['fact:a'])
cases=[({'status':'ok','text':'{"answer":0.25,"sources":["fact:a"]}'},True,'correct_answer'),({'status':'ok','text':'{"answer":0.35,"sources":["fact:a"]}'},True,'wrong_answer'),({'status':'ok','text':'{"answer":null,"sources":[]}'},False,'correct_abstention'),({'status':'ok','text':'{"answer":true,"sources":[]}'},True,'unscorable')]
for rec,access,want in cases:
 score=answer_score(rec,task,access)
 assert score['verdict']==want
 print(want,score)

correct_answer {'verdict': 'correct_answer', 'provenance_correct': True, 'parsed': {'answer': 0.25, 'sources': ['fact:a']}}
wrong_answer {'verdict': 'wrong_answer', 'provenance_correct': True, 'parsed': {'answer': 0.35, 'sources': ['fact:a']}}
correct_abstention {'verdict': 'correct_abstention', 'provenance_correct': True, 'parsed': {'answer': None, 'sources': []}}
unscorable {'verdict': 'unscorable', 'reason': ''}


**Read the output.** Each case receives its intended verdict. The wrong answer 0.35 is `wrong_answer` even though it cites the correct source, so provenance and correctness are scored separately. The Boolean answer is `unscorable` because `true` is not a number, which prevents a malformed response from being counted as correct or as an abstention. These are constructed checks of the scorer; its live verdicts follow in the next notebooks.